# Fine-tuning BART-large-XSum on MeetingBank

# MeetingBank → BART fine-tuning (training and evaluation notebook)

**Data:** huuuyeah/meetingbank (CC BY-NC-SA 4.0, non-commercial use). Cite: Hu et al., "MeetingBank: A Benchmark Dataset for Meeting Summarization", ACL 2023.
**Base model:** facebook/bart-large-xsum (check its model card for licence terms).
**Note on licences:** the dataset is CC BY-NC-SA 4.0; consider this when redistributing models fine-tuned on it.

In [ ]:
# Colab's preinstalled torch is kept as-is. Only the HF stack is pinned.
# Versions used for the training run (transformers 4.46.3). Evaluation later used transformers 4.57.3 (see evaluation section).
!pip install -q transformers==4.46.3 tokenizers==0.20.3 datasets==3.2.0 accelerate==1.2.1 evaluate==0.4.3 rouge_score==0.1.2

In [ ]:
# seeds, device detection, directories
# NOTE: the version printout recorded below is from a later session than the training run.
# Training-time versions: transformers 4.46.3, datasets 3.2.0, accelerate 1.2.1 (see the compatibility check output and the evaluation section).

import os, sys, json, random, platform
import numpy as np, torch, transformers, datasets, accelerate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    p = torch.cuda.get_device_properties(0)
    device_name = p.name
    print(f"GPU: {p.name} | {p.total_memory/1024**3:.1f} GB | CC {p.major}.{p.minor}")
else:
    p = None
    device_name = "CPU"
    print("GPU unavailable. Running on CPU.")

transformers.set_seed(SEED)

BASE = "/content"
DIRS = {k: f"{BASE}/{k}" for k in ["data_cache", "checkpoints", "final_model", "results", "logs"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

env = {
    "seed": SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "gpu": device_name if p is not None else None,
    "device": "cuda" if p is not None else "cpu",
}
json.dump(env, open(f"{DIRS['logs']}/env.json", "w"), indent=2)
print(json.dumps(env, indent=2))
print("Dirs:", DIRS)

GPU: Tesla T4 | 14.6 GB | CC 7.5
{
  "seed": 42,
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "transformers": "5.20.1",
  "datasets": "4.8.5",
  "accelerate": "1.15.0",
  "gpu": "Tesla T4",
  "device": "cuda"
}
Dirs: {'data_cache': '/content//data_cache', 'checkpoints': '/content//checkpoints', 'final_model': '/content//final_model', 'results': '/content//results', 'logs': '/content//logs'}


## 1. Data cleaning and prep

In [ ]:
# dataset metadata, license, and load

from huggingface_hub import dataset_info
from datasets import load_dataset

DATASET_ID = "huuuyeah/meetingbank"
info = dataset_info(DATASET_ID)
print("License (hub metadata):", info.card_data.license if info.card_data else "n/a")
print("Revision SHA:", info.sha)

ds = load_dataset(DATASET_ID, revision=info.sha, cache_dir=DIRS["data_cache"])
print(ds)
for s in ds:
    print(s, "->", ds[s].features)

json.dump({"dataset": DATASET_ID, "revision": info.sha, "license": info.card_data.license if info.card_data else None},
          open(f"{DIRS['logs']}/dataset_version.json", "w"), indent=2)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


License (hub metadata): cc-by-nc-sa-4.0
Revision SHA: 5b4fb6f67f490be93e249d70b732780932d19fe3


README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/88.4M [00:00<?, ?B/s]

validation.json:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/13.4M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 5169
    })
    validation: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 861
    })
    test: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 862
    })
})
train -> {'summary': Value(dtype='string', id=None), 'uid': Value(dtype='string', id=None), 'id': Value(dtype='int64', id=None), 'transcript': Value(dtype='string', id=None)}
validation -> {'summary': Value(dtype='string', id=None), 'uid': Value(dtype='string', id=None), 'id': Value(dtype='int64', id=None), 'transcript': Value(dtype='string', id=None)}
test -> {'summary': Value(dtype='string', id=None), 'uid': Value(dtype='string', id=None), 'id': Value(dtype='int64', id=None), 'transcript': Value(dtype='string', id=None)}


In [ ]:
# example

ex = ds["train"][0]
print({k: (v[:300] + "…" if isinstance(v, str) and len(v) > 300 else v) for k, v in ex.items()})

{'summary': 'AS AMENDED a bill for an ordinance amending the Denver Zoning Code to revise parking exemptions for pre-existing small zone lots. Approves a text amendment to the Denver Zoning Code to revise the Pre-Existing Small Zone Lot parking exemption. The Committee approved filing this bill at its meeting on…', 'uid': 'DenverCityCouncil_05012017_17-0161', 'id': 0, 'transcript': 'Please refrain from profane or obscene speech. Direct your comments to council as a whole and refrain from individual or personal attacks. Councilwoman Gilmore, will you please put Council Bill 161 on the floor? Yes, President Brooks, I move that council bill 161 as amended, be placed upon final con…'}


In [ ]:
# split sizes, missing values, empty strings

import pandas as pd

dfs = {s: ds[s].to_pandas() for s in ds}
report = {}

for s, df in dfs.items():
    empties = {c: int((df[c].astype(str).str.strip() == "").sum()) for c in ["transcript", "summary"]}
    report[s] = {"rows": len(df), "columns": list(df.columns), "nulls": df.isna().sum().to_dict(), "empty": empties}
    print(s, report[s])

train {'rows': 5169, 'columns': ['summary', 'uid', 'id', 'transcript'], 'nulls': {'summary': 0, 'uid': 0, 'id': 0, 'transcript': 0}, 'empty': {'transcript': 0, 'summary': 0}}
validation {'rows': 861, 'columns': ['summary', 'uid', 'id', 'transcript'], 'nulls': {'summary': 0, 'uid': 0, 'id': 0, 'transcript': 0}, 'empty': {'transcript': 0, 'summary': 0}}
test {'rows': 862, 'columns': ['summary', 'uid', 'id', 'transcript'], 'nulls': {'summary': 0, 'uid': 0, 'id': 0, 'transcript': 0}, 'empty': {'transcript': 0, 'summary': 0}}


In [ ]:
# duplicates within each split

for s, df in dfs.items():
    d = {
        "dup_uid": int(df["uid"].duplicated().sum()),
        "dup_transcript": int(df["transcript"].duplicated().sum()),
        "dup_summary": int(df["summary"].duplicated().sum()),
        "dup_transcript_summary_pair": int(df.duplicated(subset=["transcript", "summary"]).sum()),
    }
    report[s]["duplicates"] = d
    print(s, d)

train {'dup_uid': 0, 'dup_transcript': 0, 'dup_summary': 426, 'dup_transcript_summary_pair': 0}
validation {'dup_uid': 0, 'dup_transcript': 0, 'dup_summary': 0, 'dup_transcript_summary_pair': 0}
test {'dup_uid': 0, 'dup_transcript': 0, 'dup_summary': 2, 'dup_transcript_summary_pair': 0}


In [ ]:
# text lengths (words and BART tokens, inspection only, no truncation)

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("facebook/bart-large-xsum")
tok.model_max_length = 10**9  # stats only; suppresses the over-length warning

def tok_lens(texts):
    return [len(x) for x in tok(list(texts))["input_ids"]]

qs = [0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
rows = []
for s, df in dfs.items():
    df["src_words"] = df["transcript"].str.split().str.len()
    df["tgt_words"] = df["summary"].str.split().str.len()
    df["src_tok"] = tok_lens(df["transcript"])
    df["tgt_tok"] = tok_lens(df["summary"])
    for col in ["src_words", "tgt_words", "src_tok", "tgt_tok"]:
        q = df[col].quantile(qs).round(0).astype(int).to_dict()
        rows.append({"split": s, "field": col, "min": int(df[col].min()), "mean": round(df[col].mean(), 1), **{f"p{int(k*100)}": v for k, v in q.items()}})
    report[s]["over_1024_src_tok_pct"] = round(float((df["src_tok"] > 1024).mean() * 100), 2)
    report[s]["over_128_tgt_tok_pct"] = round(float((df["tgt_tok"] > 128).mean() * 100), 2)
    report[s]["median_compression_ratio"] = round(float((df["src_tok"] / df["tgt_tok"]).median()), 1)

pd.set_option("display.width", 200)
print(pd.DataFrame(rows).to_string(index=False))
for s in dfs:
    print(s, {k: report[s][k] for k in ["over_1024_src_tok_pct", "over_128_tgt_tok_pct", "median_compression_ratio"]})

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

     split     field  min   mean  p50  p75  p90   p95   p99  p100
     train src_words   85 2964.9 1036 3044 7626 12347 27700 67634
     train tgt_words   14   59.8   52   79  109   133   166   217
     train   src_tok  117 3526.2 1241 3625 9005 14732 33077 79942
     train   tgt_tok   18   86.3   74  113  158   190   249   567
validation src_words   74 2647.0  825 2511 7200 10915 24154 56330
validation tgt_words   14   59.8   52   78  110   131   166   187
validation   src_tok  107 3153.2 1002 2994 8446 12783 28892 67688
validation   tgt_tok   18   86.4   75  111  157   188   238   490
      test src_words   69 2687.5  892 2737 6480 10316 26191 63834
      test tgt_words   14   56.7   48   75  108   126   163   192
      test   src_tok  103 3193.6 1080 3237 7685 12172 31184 74873
      test   tgt_tok   18   82.5   68  107  156   186   251   550
train {'over_1024_src_tok_pct': 54.87, 'over_128_tgt_tok_pct': 18.71, 'median_compression_ratio': 17.6}
validation {'over_1024_src_tok_pct': 4

In [ ]:
# cross-split leakage checks

import re

def norm(t): return re.sub(r"\s+", " ", t.lower()).strip()

# Assumption to verify from the printed samples: uid = <City>_<date>_<item>, so meeting key = first two parts.
def meeting_key(uid): return "_".join(uid.split("_")[:2])

for s, df in dfs.items():
    df["meeting"] = df["uid"].map(meeting_key)
    df["t_norm"] = df["transcript"].map(norm)
    df["s_norm"] = df["summary"].map(norm)
    print(s, "sample uid -> meeting:", list(zip(df["uid"].head(3), df["meeting"].head(3))))

pairs = [("train", "validation"), ("train", "test"), ("validation", "test")]
leak = {}
for a, b in pairs:
    A, B = dfs[a], dfs[b]
    leak[f"{a}|{b}"] = {
        "shared_uid": len(set(A["uid"]) & set(B["uid"])),
        "shared_meeting": len(set(A["meeting"]) & set(B["meeting"])),
        "shared_transcript_exact": len(set(A["t_norm"]) & set(B["t_norm"])),
        "shared_summary_exact": len(set(A["s_norm"]) & set(B["s_norm"])),
        "shared_transcript_first200chars": len(set(A["t_norm"].str[:200]) & set(B["t_norm"].str[:200])),
    }
    print(f"{a} vs {b}:", leak[f"{a}|{b}"])
for s in dfs:
    print(s, "unique meetings:", dfs[s]["meeting"].nunique())
report["leakage"] = leak

train sample uid -> meeting: [('DenverCityCouncil_05012017_17-0161', 'DenverCityCouncil_05012017'), ('DenverCityCouncil_04102017_17-0161', 'DenverCityCouncil_04102017'), ('DenverCityCouncil_02272017_17-0161', 'DenverCityCouncil_02272017')]
validation sample uid -> meeting: [('LongBeachCC_10102017_17-0890', 'LongBeachCC_10102017'), ('SeattleCityCouncil_03142016_CB 118618', 'SeattleCityCouncil_03142016'), ('DenverCityCouncil_10112021_21-0944', 'DenverCityCouncil_10112021')]
test sample uid -> meeting: [('SeattleCityCouncil_06132016_Res 31669', 'SeattleCityCouncil_06132016'), ('BostonCC_12152021_2021-1239', 'BostonCC_12152021'), ('LongBeachCC_04072015_15-0296', 'LongBeachCC_04072015')]
train vs validation: {'shared_uid': 0, 'shared_meeting': 535, 'shared_transcript_exact': 0, 'shared_summary_exact': 12, 'shared_transcript_first200chars': 2}
train vs test: {'shared_uid': 0, 'shared_meeting': 529, 'shared_transcript_exact': 0, 'shared_summary_exact': 15, 'shared_transcript_first200chars': 3

In [ ]:
# save the inspection report to the local results directory

json.dump(report, open(f"{DIRS['results']}/section2_data_inspection.json", "w"), indent=2, default=str)
print("Saved:", f"{DIRS['results']}/section2_data_inspection.json")

Saved: /content//results/section2_data_inspection.json


In [ ]:
# UID check, full leakage report, meeting-disjoint feasibility

import pandas as pd
pd.set_option("display.width", 250, "display.max_columns", 30)

# UID format check
for s, df in dfs.items():
    parts = df["uid"].str.split("_")
    print(s, "n_parts:", parts.str.len().value_counts().to_dict(),
          "| date field is 8 digits:", round(float(parts.str[1].str.fullmatch(r"\d{8}").mean()), 4))

# Full leakage table: A = earlier split, B = later split
rows = []
for a, b in pairs:
    A, B = dfs[a], dfs[b]
    ma, mb = set(A.meeting), set(B.meeting)
    rows.append({
        "A|B": f"{a}|{b}",
        "B_meetings": len(mb), "shared_meetings": len(ma & mb),
        "B_ex_in_shared_mtg": int(B.meeting.isin(ma).sum()), "B_ex": len(B),
        "shared_uid": len(set(A.uid) & set(B.uid)),
        "shared_norm_transcript": len(set(A.t_norm) & set(B.t_norm)),
        "shared_norm_pair": len(set(zip(A.t_norm, A.s_norm)) & set(zip(B.t_norm, B.s_norm))),
        "B_summaries_exact_in_A": int(B.s_norm.isin(set(A.s_norm)).sum()),
    })
print(pd.DataFrame(rows).to_string(index=False))

# Concrete examples: 3 shared meetings (train vs test), deterministic order
A, B = dfs["train"], dfs["test"]
for m in sorted(set(A.meeting) & set(B.meeting))[:3]:
    ra, rb = A[A.meeting == m], B[B.meeting == m]
    print(f"\n== {m} | train items: {len(ra)} | test items: {len(rb)}")
    print(" train uids:", list(ra.uid[:3]))
    print(" test  uids:", list(rb.uid[:3]))
    print(" train summary:", ra.summary.iloc[0][:130])
    print(" test  summary:", rb.summary.iloc[0][:130])

# Test summaries that also appear verbatim in train
dupmask = B.s_norm.isin(set(A.s_norm))
print("\nTest summaries appearing verbatim in train:", int(dupmask.sum()))
for _, r in B[dupmask].head(3).iterrows():
    print(" -", r.uid, "|", r.summary[:120])

# Most repeated train summaries
print("\nTop repeated train summaries:")
print(A.s_norm.value_counts().head(5).to_string())

# Feasibility of the meeting-disjoint protocol (uses test meeting keys only, never test content)
test_mtg = set(dfs["test"].meeting)
pool = pd.concat([dfs["train"], dfs["validation"]])
keep = pool[~pool.meeting.isin(test_mtg)]
print(f"\nTrain+val pool: {len(pool)} | kept after removing meetings in official test: {len(keep)} "
      f"({100*len(keep)/len(pool):.1f}%) across {keep.meeting.nunique()} meetings")
print("Official test examples whose meeting is absent from train+val:",
      int((~dfs['test'].meeting.isin(set(pool.meeting))).sum()), "of", len(dfs["test"]))

train n_parts: {3: 5169} | date field is 8 digits: 1.0
validation n_parts: {3: 861} | date field is 8 digits: 1.0
test n_parts: {3: 861, 4: 1} | date field is 8 digits: 0.9988
             A|B  B_meetings  shared_meetings  B_ex_in_shared_mtg  B_ex  shared_uid  shared_norm_transcript  shared_norm_pair  B_summaries_exact_in_A
train|validation         560              535                 832   861           0                       0                 0                      12
      train|test         559              529                 831   862           0                       0                 0                      16
 validation|test         559              291                 485   862           0                       0                 0                       4

== AlamedaCC_01052021 | train items: 1 | test items: 1
 train uids: ['AlamedaCC_01052021_2021-8482']
 test  uids: ['AlamedaCC_01052021_2021-8445']
 train summary: Introduction of Ordinance Authorizing the City Manager to Ex

In [ ]:
# length options, summary shape, generation defaults

import numpy as np, re
from transformers import AutoConfig, GenerationConfig

rows = []
for s, df in dfs.items():
    for kind, col, limits in [("src", "src_tok", [512, 1024]), ("tgt", "tgt_tok", [128, 192, 256])]:
        for L in limits:
            frac = np.minimum(df[col], L) / df[col]
            rows.append({"split": s, "limit": f"{kind}@{L}", "fit": int((df[col] <= L).sum()),
                         "truncated": int((df[col] > L).sum()),
                         "median_frac_kept": round(float(frac.median()), 3),
                         "mean_frac_kept": round(float(frac.mean()), 3)})
print(pd.DataFrame(rows).to_string(index=False))

for s, df in dfs.items():
    print(s, "fit src<=1024 AND tgt<=256:", int(((df.src_tok <= 1024) & (df.tgt_tok <= 256)).sum()), "of", len(df))

# How many sentences do reference summaries have? (relevant to the XSum prior)
n_sent = dfs["train"].summary.map(lambda t: len(re.split(r"(?<=[.!?])\s+", t.strip())))
print("\nTrain summary sentence counts (share):", n_sent.value_counts(normalize=True).sort_index().round(3).head(6).to_dict())
for t in dfs["train"].summary.sample(3, random_state=SEED):
    print(" -", t[:200])

# Generation defaults + position limit of the provisional checkpoint (config files only)
cfg = AutoConfig.from_pretrained("facebook/bart-large-xsum")
print("\nmax_position_embeddings:", cfg.max_position_embeddings)
try:
    print(GenerationConfig.from_pretrained("facebook/bart-large-xsum").to_dict())
except Exception as e:
    print("no generation_config.json; model config values:",
          {k: getattr(cfg, k, None) for k in ["max_length", "min_length", "num_beams", "length_penalty", "no_repeat_ngram_size", "early_stopping"]})

     split    limit  fit  truncated  median_frac_kept  mean_frac_kept
     train  src@512 1420       3749             0.413           0.508
     train src@1024 2333       2836             0.825           0.660
     train  tgt@128 4202        967             1.000           0.957
     train  tgt@192 4925        244             1.000           0.992
     train  tgt@256 5127         42             1.000           0.998
validation  src@512  276        585             0.511           0.552
validation src@1024  435        426             1.000           0.697
validation  tgt@128  706        155             1.000           0.959
validation  tgt@192  821         40             1.000           0.993
validation  tgt@256  856          5             1.000           0.998
      test  src@512  253        609             0.474           0.531
      test src@1024  416        446             0.948           0.682
      test  tgt@128  717        145             1.000           0.960
      test  tgt@192 

generation_config.json:   0%|          | 0.00/309 [00:00<?, ?B/s]

{'max_length': 62, 'max_new_tokens': None, 'min_length': 11, 'min_new_tokens': None, 'early_stopping': True, 'max_time': None, 'stop_strings': None, 'do_sample': False, 'num_beams': 6, 'num_beam_groups': 1, 'penalty_alpha': None, 'dola_layers': None, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'temperature': 1.0, 'top_k': 50, 'top_p': 1.0, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'diversity_penalty': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3, 'bad_words_ids': None, 'force_words_ids': None, 'renormalize_logits': False, 'constraints': None, 'forced_bos_token_id': None, 'forced_eos_token_id': 2, 'remove_invalid_values': False, 'exponential_decay_length_penalty': None, 'suppress_tokens': None, 'begin_suppress_tokens': None, 'forced_decoder_ids': None, 'sequence_bias': None, 'token_healing': False, 'guidance_scale': None, 'low_me

In [ ]:
# training-stack compatibility check (no model download, no reinstall)

import importlib

for m in [
    "torch", "transformers", "tokenizers", "datasets", "accelerate",
    "evaluate", "rouge_score", "huggingface_hub", "numpy", "pandas"
]:
    print(m, getattr(importlib.import_module(m), "__version__", "?"))

from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    AutoModelForSeq2SeqLM,
)
import evaluate

rouge = evaluate.load("rouge")
print(
    rouge.compute(
        predictions=["the cat sat on the mat"],
        references=["the cat is on the mat"],
        use_stemmer=True,
    )
)

has_cuda = torch.cuda.is_available()

args = Seq2SeqTrainingArguments(
    output_dir="/tmp/_chk",
    fp16=has_cuda,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    report_to="none",
)

print(
    "TrainingArguments OK | fp16:",
    args.fp16,
    "| device:",
    args.device,
)

device = torch.device("cuda" if has_cuda else "cpu")
lin = torch.nn.Linear(8, 8).to(device)
opt = torch.optim.AdamW(lin.parameters())

if has_cuda:
    scaler = torch.amp.GradScaler("cuda")

    with torch.autocast("cuda", dtype=torch.float16):
        loss = lin(torch.randn(4, 8, device=device)).float().pow(2).mean()

    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
else:
    loss = lin(torch.randn(4, 8, device=device)).pow(2).mean()
    loss.backward()
    opt.step()

print("Optimizer step OK | device:", device)

torch 2.11.0+cu130
transformers 4.46.3
tokenizers 0.20.3
datasets 3.2.0
accelerate 1.2.1
evaluate 0.4.3
rouge_score ?
huggingface_hub 0.36.2
numpy 2.1.3
pandas 2.2.3


{'rouge1': np.float64(0.8333333333333334), 'rouge2': np.float64(0.6), 'rougeL': np.float64(0.8333333333333334), 'rougeLsum': np.float64(0.8333333333333334)}
TrainingArguments OK | fp16: True | device: cuda:0
Optimizer step OK | device: cuda


In [ ]:
# Test summaries that also occur in train: are the (transcript, summary) pairs identical? Read-only; nothing is modified.
A, B = dfs["train"], dfs["test"]
s_counts = A.s_norm.value_counts()
s_meetings = A.groupby("s_norm")["meeting"].agg(set)
pair_set = set(zip(A.t_norm, A.s_norm))

m = B[B.s_norm.isin(set(A.s_norm))].copy()
m["n_train_same_summary"] = m.s_norm.map(s_counts)
m["identical_pair_in_train"] = [(t, s) in pair_set for t, s in zip(m.t_norm, m.s_norm)]
m["same_meeting_as_a_match"] = [mt in s_meetings[s] for mt, s in zip(m.meeting, m.s_norm)]
m["summary_head"] = m.summary.str[:70]

print("Test examples with a summary also in train:", len(m))
print("  identical transcript+summary pair in train:", int(m.identical_pair_in_train.sum()))
print("  summary match comes from the same meeting:", int(m.same_meeting_as_a_match.sum()))
print(m[["uid", "n_train_same_summary", "identical_pair_in_train", "same_meeting_as_a_match", "src_tok", "summary_head"]].to_string(index=False))
m.drop(columns=["t_norm", "s_norm", "transcript"]).to_json(f"{DIRS['results']}/test_summary_overlap.json", orient="records", indent=2)

Test examples with a summary also in train: 16
  identical transcript+summary pair in train: 0
  summary match comes from the same meeting: 0
                                  uid  n_train_same_summary  identical_pair_in_train  same_meeting_as_a_match  src_tok                                                           summary_head
SeattleCityCouncil_12142020_CB 119971                     1                    False                    False     1158 AN ORDINANCE authorizing, in 2020, acceptance of funding from non-City
         LongBeachCC_05102022_22-0507                     3                    False                    False     1015 Recommendation to receive supporting documentation into the record, co
   DenverCityCouncil_05132019_19-0473                     3                    False                    False     2119 A proclamation recognizing May 19th as “Hepatitis Testing Day” and Jul
         LongBeachCC_03132018_18-0219                     2                    False              

**Limitations of this evaluation (official splits kept):** MeetingBank splits by agenda item, not by meeting, so most validation/test meetings also appear in train. Test scores measure generalization to held-out items from known cities and meetings, and may be optimistic versus unseen meetings. Some summaries repeat verbatim across splits. Inputs are truncated to 1,024 tokens; a large share of test sources exceed this limit, so for those items the model never sees the end of the meeting (see the test-evaluation cell below).

In [ ]:
# config and training args
# Optimizer and LR scheduler are not set explicitly: Hugging Face Trainer defaults apply (AdamW, linear decay).

from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
from transformers.trainer_utils import get_last_checkpoint

USE_CUDA = torch.cuda.is_available()
CONFIG = {
    "model_name": "facebook/bart-large-xsum", "dataset": DATASET_ID, "dataset_revision": info.sha, "seed": SEED,
    "max_source_len": 1024, "max_target_len": 256,
    "epochs": 4, "learning_rate": 2e-5, "weight_decay": 0.01, "warmup_ratio": 0.05,
    "per_device_train_batch_size": 1, "gradient_accumulation_steps": 8, "per_device_eval_batch_size": 2,
    "gradient_checkpointing": True, "fp16": USE_CUDA, "selection_metric": "eval_loss", "save_total_limit": 2,
}
training_args = Seq2SeqTrainingArguments(
    output_dir=DIRS["checkpoints"],
    num_train_epochs=CONFIG["epochs"], learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"], warmup_ratio=CONFIG["warmup_ratio"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=USE_CUDA, predict_with_generate=False,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=CONFIG["save_total_limit"],
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=50, report_to="none", seed=SEED, data_seed=SEED,
)
json.dump(CONFIG, open(f"{DIRS['logs']}/training_config.json", "w"), indent=2)
print("TrainingArguments OK | fp16:", training_args.fp16, "| device:", training_args.device)

TrainingArguments OK | fp16: True | device: cuda:0


In [ ]:
# integrity checks, tokenization, truncation statistics

import numpy as np, pandas as pd

for s, df in dfs.items():  # fail loudly on integrity problems
    assert not df[["transcript", "summary"]].isna().any().any(), f"nulls in {s}"
    assert not (df.transcript.str.strip() == "").any() and not (df.summary.str.strip() == "").any(), f"empty text in {s}"

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])  # fresh: 2.3 modified `tok`
print("tokenizer.model_max_length:", tokenizer.model_max_length)

def preprocess(batch):
    enc = tokenizer(batch["transcript"], max_length=CONFIG["max_source_len"], truncation=True)
    enc["labels"] = tokenizer(text_target=batch["summary"], max_length=CONFIG["max_target_len"], truncation=True)["input_ids"]
    return enc

tokenized = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names, desc="Tokenizing")

eos, rows = tokenizer.eos_token_id, []
for s in tokenized:
    src = np.array([len(x) for x in tokenized[s]["input_ids"]]); tgt = np.array([len(x) for x in tokenized[s]["labels"]])
    fs, ft = dfs[s]["src_tok"].to_numpy(), dfs[s]["tgt_tok"].to_numpy()
    assert (src == np.minimum(fs, CONFIG["max_source_len"])).all() and (tgt == np.minimum(ft, CONFIG["max_target_len"])).all()
    assert all(x[-1] == eos for x in tokenized[s]["input_ids"]) and all(x[-1] == eos for x in tokenized[s]["labels"])
    rows.append({"split": s, "n": len(src),
                 "src_truncated": int((fs > CONFIG["max_source_len"]).sum()), "src_trunc_pct": round(100 * float((fs > CONFIG["max_source_len"]).mean()), 1),
                 "src_tokens_kept_pct": round(100 * float(src.sum() / fs.sum()), 1),
                 "tgt_truncated": int((ft > CONFIG["max_target_len"]).sum()), "tgt_trunc_pct": round(100 * float((ft > CONFIG["max_target_len"]).mean()), 2),
                 "tgt_tokens_kept_pct": round(100 * float(tgt.sum() / ft.sum()), 2)})
stats = pd.DataFrame(rows)
print(stats.to_string(index=False))
stats.to_json(f"{DIRS['results']}/truncation_stats.json", orient="records", indent=2)
print("\nDecoded label[0]:", tokenizer.decode(tokenized["train"][0]["labels"])[:200])

tokenizer.model_max_length: 1024


Tokenizing:   0%|          | 0/5169 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/861 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/862 [00:00<?, ? examples/s]

     split    n  src_truncated  src_trunc_pct  src_tokens_kept_pct  tgt_truncated  tgt_trunc_pct  tgt_tokens_kept_pct
     train 5169           2836           54.9                 22.1             42           0.81                99.10
validation  861            426           49.5                 23.7              5           0.58                99.03
      test  862            446           51.7                 23.8              6           0.70                98.69

Decoded label[0]: <s>AS AMENDED a bill for an ordinance amending the Denver Zoning Code to revise parking exemptions for pre-existing small zone lots. Approves a text amendment to the Denver Zoning Code to revise the P


In [ ]:
# GPU preflight and smoke test

import gc, shutil

def gpu_preflight():
    ok = torch.cuda.is_available()
    print("CUDA available:", ok)
    if ok:
        free, total = torch.cuda.mem_get_info()
        print(f"GPU: {torch.cuda.get_device_name(0)} | free {free/1024**3:.1f} / {total/1024**3:.1f} GiB")
        print(f"Drive free (may be approximate): {shutil.disk_usage(BASE).free/1024**3:.1f} GiB")
    return ok

def run_smoke_test():
    if not gpu_preflight():
        print("SKIPPED: no GPU. SMOKE_PASSED stays False."); return False
    ckpt = "/content/_smoke_ckpt"
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"]).to("cuda")
        print("BART-large loaded.")
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False}); model.config.use_cache = False
        collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

        d = tokenized["train"]  # worst case for memory: longest source + target combination
        idx = int(np.argmax([len(a) + len(b) for a, b in zip(d["input_ids"], d["labels"])]))
        print(f"Worst-case example: src={len(d[idx]['input_ids'])} tgt={len(d[idx]['labels'])}")
        batch = {k: v.to("cuda") for k, v in collator([d[idx]]).items()}

        opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"]); scaler = torch.amp.GradScaler("cuda")
        model.train()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**batch).loss
        assert torch.isfinite(loss), f"non-finite loss: {loss.item()}"
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); torch.cuda.synchronize()
        print(f"Optimizer step OK | loss {loss.item():.3f} | peak GPU mem {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB")

        model.save_pretrained(ckpt); model.eval()
        with torch.no_grad():
            ref = model(**batch).logits.float()
        del model, opt; gc.collect(); torch.cuda.empty_cache()
        re = AutoModelForSeq2SeqLM.from_pretrained(ckpt).to("cuda").eval()
        with torch.no_grad():
            diff = (re(**batch).logits.float() - ref).abs().max().item()
        print(f"Checkpoint reload OK | max logit diff {diff:.2e}")
        assert diff < 1e-3
        del re; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(ckpt, ignore_errors=True)
        return True
    except torch.cuda.OutOfMemoryError as e:
        print("OOM at worst-case lengths:", str(e)[:200]); return False

SMOKE_PASSED = run_smoke_test()
print("SMOKE_PASSED:", SMOKE_PASSED)

CUDA available: True
GPU: Tesla T4 | free 14.4 / 14.6 GiB
Drive free (may be approximate): 69.8 GiB


pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

BART-large loaded.
Worst-case example: src=1024 tgt=256
Optimizer step OK | loss 3.456 | peak GPU mem 3.24 GiB


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 62, 'min_length': 11, 'early_stopping': True, 'num_beams': 6, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Checkpoint reload OK | max logit diff 0.00e+00
SMOKE_PASSED: True


In [ ]:
# training

assert SMOKE_PASSED, "Run the smoke-test cell above successfully on a GPU first."
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"]); model.config.use_cache = False
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
trainer = Seq2SeqTrainer(model=model, args=training_args, train_dataset=tokenized["train"],
                         eval_dataset=tokenized["validation"], data_collator=collator, tokenizer=tokenizer)
last = get_last_checkpoint(DIRS["checkpoints"]); print("Resuming from:", last)
trainer.train(resume_from_checkpoint=last)

trainer.save_model(DIRS["final_model"]); tokenizer.save_pretrained(DIRS["final_model"])
json.dump(trainer.state.log_history, open(f"{DIRS['logs']}/log_history.json", "w"), indent=2)
json.dump({"best_checkpoint": trainer.state.best_model_checkpoint, "best_eval_loss": trainer.state.best_metric},
          open(f"{DIRS['results']}/best_checkpoint.json", "w"), indent=2)
print("Best:", trainer.state.best_model_checkpoint, trainer.state.best_metric)

/tmp/ipykernel_527/3337253313.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(model=model, args=training_args, train_dataset=tokenized["train"],


Resuming from: None


Epoch,Training Loss,Validation Loss
0,1.161200,1.129178
1,0.829100,1.058267
2,0.713400,1.020616
3,0.544500,1.048229


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 62, 'min_length': 11, 'early_stopping': True, 'num_beams': 6, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Best: /content//checkpoints/checkpoint-1938 1.0206164121627808


> **Training note:** validation loss was lowest at epoch 3 (`checkpoint-1938`, 1.0206) and rose at epoch 4 (`checkpoint-2584`, 1.0482), so the epoch-3 checkpoint was kept as best rather than the final one.

> **Where artifacts live:** training ran in `/content` and checkpoints were afterwards backed up to persistent storage. The evaluation section below loads from `ARTIFACT_DIR` (set it in the first evaluation code cell to your own backup folder).

## Evaluation

**Protocol:** the checkpoint was selected by validation loss (`checkpoint-1938`); the official test split is evaluated once with a generation config fixed beforehand (see the generation-config cell). The evaluation runtime uses transformers 4.57.3.


In [ ]:
# Evaluation environment: the checkpoint failed to load under transformers 5.20.1; evaluation used 4.57.3.
# Restart the runtime after this install, then run the evaluation cells.
%pip install -q "transformers==4.57.3"


In [ ]:
# paths + dependency check. Read-only except the helper's own eval_* output files.
import json, hashlib, re, time
from pathlib import Path
import numpy as np, torch, transformers, datasets, accelerate, evaluate
import rouge_score  # import check only

# Set this to the folder containing the training backup (subfolders: checkpoints/, logs/, results/, data_cache/).
ARTIFACT_DIR = Path("/path/to/meetingbank_bart_backup")
BACKUP_DIR = ARTIFACT_DIR
CKPT_DIR, LOGS_DIR, RESULTS = BACKUP_DIR / "checkpoints", BACKUP_DIR / "logs", BACKUP_DIR / "results"
for p in (BACKUP_DIR, CKPT_DIR, LOGS_DIR, RESULTS):
    assert p.is_dir(), f"missing: {p}"
assert torch.cuda.is_available(), "GPU required"

CFG = json.load(open(LOGS_DIR / "training_config.json"))
env_train = json.load(open(LOGS_DIR / "env.json"))
now = {"torch": torch.__version__, "transformers": transformers.__version__,
       "datasets": datasets.__version__, "accelerate": accelerate.__version__}
for k, v in now.items():
    print(f"{k:13s} training={env_train.get(k)}  now={v}", "" if env_train.get(k) == v else "  <-- MISMATCH")
print("evaluate", evaluate.__version__)

if "ds" not in globals():  # minimal restoration: pinned revision, from the backup's own cache, no download
    from datasets import load_dataset, DownloadConfig
    ver = json.load(open(LOGS_DIR / "dataset_version.json"))
    ds = load_dataset(ver["dataset"], revision=ver["revision"], cache_dir=str(BACKUP_DIR / "data_cache"),
                      download_config=DownloadConfig(local_files_only=True))
sizes = {k: ds[k].num_rows for k in ds}; print("splits:", sizes)
assert sizes == {"train": 5169, "validation": 861, "test": 862}
assert {"uid", "transcript", "summary"} <= set(ds["validation"].column_names)
assert CFG["max_source_len"] == 1024 and CFG["max_target_len"] == 256 and CFG["model_name"] == "facebook/bart-large-xsum"
SEED, MAX_SRC, MAX_TGT = CFG["seed"], CFG["max_source_len"], CFG["max_target_len"]
transformers.set_seed(SEED)

def save_json(name, obj, overwrite=False):
    p = RESULTS / name
    if p.exists() and not overwrite:
        raise FileExistsError(f"{p} exists; refusing to overwrite")
    with open(p, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    return p
print("E0 OK")

torch         training=2.11.0+cu130  now=2.11.0+cu130 
transformers  training=4.46.3  now=4.57.3   <-- MISMATCH
datasets      training=3.2.0  now=3.2.0 
accelerate    training=1.2.1  now=1.2.1 
evaluate 0.4.3


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


splits: {'train': 5169, 'validation': 861, 'test': 862}
E0 OK


> The `embed_tokens` / `lm_head` missing-keys warning printed at the end of training is expected to relate to tied weights; the cell below verifies the saved best checkpoint loads and reproduces the logged validation loss.

In [ ]:
# checkpoint metadata, files, final_model status, missing-keys investigation. Sets ARTIFACT_FILES_OK.
from safetensors import safe_open
from transformers import AutoModelForSeq2SeqLM
import json

def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]

EXPECTED_BEST, EXPECTED_LOSS = "checkpoint-1938", 1.020616
bj = json.load(open(RESULTS / "best_checkpoint.json")); print("best_checkpoint.json:", bj)
best_name = Path(bj["best_checkpoint"]).name   # name only; the recorded /content path is never used
assert best_name == EXPECTED_BEST and abs(bj["best_eval_loss"] - EXPECTED_LOSS) < 1e-6, bj
CKPT_BEST = CKPT_DIR / best_name

ts_last = json.load(open(CKPT_DIR / "checkpoint-2584" / "trainer_state.json"))
ts_best = json.load(open(CKPT_BEST / "trainer_state.json"))
evals = [(e["step"], round(e["epoch"], 2), e["eval_loss"]) for e in ts_last["log_history"] if "eval_loss" in e]
print("eval history (step, epoch, eval_loss):", evals)
assert Path(ts_last["best_model_checkpoint"]).name == best_name and abs(ts_last["best_metric"] - EXPECTED_LOSS) < 1e-6
assert ts_best["global_step"] == 1938 and min(evals, key=lambda e: e[2])[0] == 1938
lh = [e["eval_loss"] for e in json.load(open(LOGS_DIR / "log_history.json")) if "eval_loss" in e]
assert np.allclose(lh, [e[2] for e in evals]), "logs/log_history.json disagrees with trainer_state.json"

for n in ["model.safetensors", "config.json", "generation_config.json", "trainer_state.json"]:
    assert (CKPT_BEST / n).is_file(), f"missing {n}"
print("best ckpt files:", {p.name: f"{p.stat().st_size/1024**2:.0f} MB" for p in sorted(CKPT_BEST.iterdir())})
assert ((CKPT_BEST / "vocab.json").exists() and (CKPT_BEST / "merges.txt").exists()) or (CKPT_BEST / "tokenizer.json").exists()

FINAL = BACKUP_DIR / "final_model"
print("\nfinal_model files:", sorted(p.name for p in FINAL.iterdir()))
print("final_model has weights:", any((FINAL / n).exists() for n in ("model.safetensors", "pytorch_model.bin")))
cb, cf = json.load(open(CKPT_BEST / "config.json")), json.load(open(FINAL / "config.json"))
print("config differences best vs final:", {k: (cb.get(k), cf.get(k)) for k in set(cb) | set(cf) if cb.get(k) != cf.get(k)})
print("tokenizer files identical:", {n: sha(CKPT_BEST / n) == sha(FINAL / n) for n in ("vocab.json", "merges.txt", "tokenizer.json")
                                     if (CKPT_BEST / n).exists() and (FINAL / n).exists()})
print("checkpoint config use_cache:", cb.get("use_cache"))

with safe_open(str(CKPT_BEST / "model.safetensors"), "pt", device="cpu") as f:
    keys = set(f.keys()); shared_file = f.get_tensor("model.shared.weight")
print("\ntensors in file:", len(keys), "| embed/lm_head keys:", sorted(k for k in keys if "embed_tokens" in k or "lm_head" in k))
model, info = AutoModelForSeq2SeqLM.from_pretrained(str(CKPT_BEST), output_loading_info=True)
print("loading info:", info)
ALLOWED = {"model.encoder.embed_tokens.weight", "model.decoder.embed_tokens.weight", "lm_head.weight"}
assert set(info["missing_keys"]) <= ALLOWED and not info["unexpected_keys"] and not info["mismatched_keys"], info
sw = model.model.shared.weight
assert torch.equal(sw.detach(), shared_file), "shared embedding differs from file"
ties = {"encoder": model.model.encoder.embed_tokens.weight.data_ptr() == sw.data_ptr(),
        "decoder": model.model.decoder.embed_tokens.weight.data_ptr() == sw.data_ptr(),
        "lm_head": model.lm_head.weight.data_ptr() == sw.data_ptr()}
print("tied to model.shared:", ties); assert all(ties.values())
ARTIFACT_FILES_OK = True; print("E1 OK")

best_checkpoint.json: {'best_checkpoint': '/content//checkpoints/checkpoint-1938', 'best_eval_loss': 1.0206164121627808}
eval history (step, epoch, eval_loss): [(646, 1.0, 1.1291780471801758), (1292, 2.0, 1.0582668781280518), (1938, 3.0, 1.0206164121627808), (2584, 4.0, 1.0482290983200073)]
best ckpt files: {'config.json': '0 MB', 'generation_config.json': '0 MB', 'merges.txt': '0 MB', 'model.safetensors': '1550 MB', 'optimizer.pt': '3100 MB', 'rng_state.pth': '0 MB', 'scheduler.pt': '0 MB', 'special_tokens_map.json': '0 MB', 'tokenizer.json': '3 MB', 'tokenizer_config.json': '0 MB', 'trainer_state.json': '0 MB', 'training_args.bin': '0 MB', 'vocab.json': '1 MB'}

final_model files: ['config.json', 'generation_config.json', 'merges.txt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.json']
final_model has weights: False
config differences best vs final: {}
tokenizer files identical: {'vocab.json': True, 'merges.txt': True, 'tokenizer.

In [ ]:
# reproduce the recorded validation loss from the Drive checkpoint. Sets ARTIFACTS_OK.
from transformers import AutoTokenizer, DataCollatorForSeq2Seq
assert ARTIFACT_FILES_OK
tokenizer = AutoTokenizer.from_pretrained(str(CKPT_BEST))   # the checkpoint's own tokenizer
print("tokenizer.model_max_length:", tokenizer.model_max_length)

def encode(split):  # same protocol as training preprocessing
    d = ds[split]
    enc = tokenizer(d["transcript"], max_length=MAX_SRC, truncation=True)
    lab = tokenizer(text_target=d["summary"], max_length=MAX_TGT, truncation=True)["input_ids"]
    return [{"input_ids": i, "attention_mask": a, "labels": l} for i, a, l in zip(enc["input_ids"], enc["attention_mask"], lab)]

model = model.to("cuda").eval()
val = encode("validation")
coll = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
losses, sizes_ = [], []
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    for i in range(0, len(val), 2):   # bs=2, as in training eval
        chunk = val[i:i + 2]
        b = {k: v.to("cuda") for k, v in coll(chunk).items()}
        losses.append(model(**b).loss.float().item()); sizes_.append(len(chunk))
val_loss = float(np.average(losses, weights=sizes_))
print(f"recomputed validation loss {val_loss:.4f} vs recorded {EXPECTED_LOSS} (diff {val_loss - EXPECTED_LOSS:+.4f})")
assert abs(val_loss - EXPECTED_LOSS) < 0.01, "does not reproduce: stop and report"
ARTIFACTS_OK = True
save_json("eval_artifact_check.json", {"checkpoint": best_name, "recorded_val_loss": EXPECTED_LOSS, "recomputed_val_loss": val_loss,
          "loading_info": info, "tied": ties, "eval_history": evals, "versions": now}, overwrite=True)
print("E2 OK")

tokenizer.model_max_length: 1024
recomputed validation loss 1.0204 vs recorded 1.020616 (diff -0.0002)
E2 OK


In [ ]:
# fixed generation config + metric helpers + 32-example VALIDATION smoke test. Sets GEN_SMOKE_OK.
from transformers import GenerationConfig
assert ARTIFACTS_OK
gen_cfg = GenerationConfig.from_pretrained(str(CKPT_BEST))
KEYS = ["num_beams", "max_length", "min_length", "length_penalty", "no_repeat_ngram_size", "early_stopping",
        "forced_bos_token_id", "forced_eos_token_id", "decoder_start_token_id", "bos_token_id", "eos_token_id", "pad_token_id", "use_cache"]
print("inherited from checkpoint:", {k: getattr(gen_cfg, k, None) for k in KEYS})
# Fixed before the test run (train targets: median ~74 tokens, p99 ~249); not tuned on the test set. The hash below guards against later changes.
# Note: max_new_tokens/min_new_tokens take precedence over the inherited max_length=62 / min_length=11.
gen_cfg.update(num_beams=4, do_sample=False, max_new_tokens=256, min_new_tokens=16,
               length_penalty=2.0, no_repeat_ngram_size=3, early_stopping=True, use_cache=True)
GEN_HASH = hashlib.sha256(json.dumps(gen_cfg.to_dict(), sort_keys=True, default=str).encode()).hexdigest()[:12]
print("FINAL generation config:", gen_cfg.to_dict(), "\nhash:", GEN_HASH)
model.config.use_cache = True   # training saved use_cache=False
GEN_BATCH = 4

def generate_texts(encoded, bs=GEN_BATCH):
    order = sorted(range(len(encoded)), key=lambda i: -len(encoded[i]["input_ids"]))  # longest first: OOM shows immediately
    out, i, cur = [None] * len(encoded), 0, bs
    while i < len(order):
        idx = order[i:i + cur]
        try:
            batch = tokenizer.pad({"input_ids": [encoded[j]["input_ids"] for j in idx]}, return_tensors="pt").to("cuda")
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                seq = model.generate(**batch, generation_config=gen_cfg)
            for j, t in zip(idx, tokenizer.batch_decode(seq, skip_special_tokens=True, clean_up_tokenization_spaces=True)):
                out[j] = t.strip()
            i += len(idx)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if cur == 1: raise
            cur = max(1, cur // 2); print("OOM -> generation batch size", cur)
    return out

rouge = evaluate.load("rouge")
def nl(t): return "\n".join(s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s)   # regex sentence split for rougeLsum
def score(preds, refs):  # deterministic mean of per-example F1 (no bootstrap), stemming on
    r = rouge.compute(predictions=[nl(p) for p in preds], references=[nl(x) for x in refs], use_stemmer=True, use_aggregator=False)
    per = {k: np.array(v) for k, v in r.items()}
    return per, {k: round(100 * float(v.mean()), 2) for k, v in per.items()}
def rep3(t):
    w = re.findall(r"\w+", t.lower()); g = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    return len(g) != len(set(g))
def diagnostics(preds, refs):
    pt = np.array([len(tokenizer(p)["input_ids"]) for p in preds]); rt = np.array([len(tokenizer(r)["input_ids"]) for r in refs])
    return {"n": len(preds), "empty_pred": int(sum(p == "" for p in preds)),
            "pred_tokens_mean/median": [round(float(pt.mean()), 1), float(np.median(pt))],
            "ref_tokens_mean/median": [round(float(rt.mean()), 1), float(np.median(rt))],
            "pred_near_max_len_pct": round(100 * float((pt >= 250).mean()), 1),
            "pred_no_final_punct_pct": round(100 * float(np.mean([p[-1:] not in ".!?" for p in preds])), 1),
            "pred_repeated_trigram_pct": round(100 * float(np.mean([rep3(p) for p in preds])), 1),
            "ref_repeated_trigram_pct": round(100 * float(np.mean([rep3(r) for r in refs])), 1)}

vrefs = ds["validation"]["summary"]
vidx = sorted(np.random.RandomState(SEED).choice(len(val), 32, replace=False).tolist())
t0 = time.time(); vp = generate_texts([val[i] for i in vidx]); dt = time.time() - t0
vr = [vrefs[i] for i in vidx]
_, vagg = score(vp, vr); vdiag = diagnostics(vp, vr)
print(f"\n32 validation examples in {dt:.0f}s ({dt/32:.1f}s/example); sanity scores (NOT a metric):", vagg)
print("diagnostics:", vdiag)
for k in range(3):
    print(f"\n[{k}] REF : {vr[k][:300]}\n    PRED: {vp[k][:300]}")
GEN_SMOKE_OK = vdiag["empty_pred"] == 0 and all(isinstance(p, str) for p in vp)
SMOKE_HASH = GEN_HASH
save_json("eval_validation_smoke.json", {"gen_config": gen_cfg.to_dict(), "hash": GEN_HASH, "indices": vidx, "scores_sanity_only": vagg,
          "diagnostics": vdiag, "predictions": vp, "references": vr}, overwrite=True)
print("\nGEN_SMOKE_OK:", GEN_SMOKE_OK)

inherited from checkpoint: {'num_beams': 6, 'max_length': 62, 'min_length': 11, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3, 'early_stopping': True, 'forced_bos_token_id': None, 'forced_eos_token_id': 2, 'decoder_start_token_id': 2, 'bos_token_id': 0, 'eos_token_id': 2, 'pad_token_id': 1, 'use_cache': True}
FINAL generation config: {'max_length': 62, 'max_new_tokens': 256, 'min_length': 11, 'min_new_tokens': 16, 'early_stopping': True, 'max_time': None, 'stop_strings': None, 'do_sample': False, 'num_beams': 4, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'prefill_chunk_size': None, 'temperature': 1.0, 'top_k': 50, 'top_p': 1.0, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'bad_words_ids': None, 'renormalize_logits': False, 'forced_bos_token_id': None, 'forced_eos_token_id': 2, 'remove_in

You're using a BartTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



32 validation examples in 25s (0.8s/example); sanity scores (NOT a metric): {'rouge1': 67.42, 'rouge2': 54.7, 'rougeL': 63.41, 'rougeLsum': 63.76}
diagnostics: {'n': 32, 'empty_pred': 0, 'pred_tokens_mean/median': [66.7, 55.0], 'ref_tokens_mean/median': [91.6, 83.0], 'pred_near_max_len_pct': 0.0, 'pred_no_final_punct_pct': 40.6, 'pred_repeated_trigram_pct': 15.6, 'ref_repeated_trigram_pct': 50.0}

[0] REF : Recommendation to request City Attorney to draft an ordinance prohibiting the use of electronically motorized boards as defined in Vehicle Code Section 313.5 from operating on sidewalks in business corridors.
    PRED: Recommendation to request City Attorney to draft an ordinance prohibiting the use of electronically motorized boards from operating on sidewalks in Business Corridors.

[1] REF : AN ORDINANCE relating to the organization of City government; moving the Office of Labor Standards from a subsidiary unit in the Office for Civil Rights to its own unit in the Executive Depa

In [ ]:
# T1: official test evaluation. RUN ONCE. Test is not used for any selection or tuning.
assert ARTIFACTS_OK and GEN_SMOKE_OK, "artifact checks / validation smoke test not passed"
assert GEN_HASH == SMOKE_HASH, "generation config changed since the validation smoke test: rerun G1"
names = ["eval_test_metrics.json", "eval_test_predictions.jsonl", "eval_generation_config.json", "eval_metadata.json", "eval_qualitative_examples.json"]
assert not any((RESULTS / n).exists() for n in names), "test outputs already exist; refusing to overwrite"

from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()
t = ds["test"]; uids, srcs, refs = t["uid"], t["transcript"], t["summary"]
full_len = np.array([len(x) for x in tokenizer(srcs, truncation=False)["input_ids"]])
enc = tokenizer(srcs, max_length=MAX_SRC, truncation=True)
test_enc = [{"input_ids": i} for i in enc["input_ids"]]
assert (np.minimum(full_len, MAX_SRC) == np.array([len(x["input_ids"]) for x in test_enc])).all()

t0 = time.time(); preds = generate_texts(test_enc); elapsed = time.time() - t0
assert len(preds) == len(refs) == 862 and all(isinstance(p, str) for p in preds)
per, agg = score(preds, refs)
fits = full_len <= MAX_SRC
strat = {"all": agg,
         f"source<={MAX_SRC} tokens (n={int(fits.sum())})": {k: round(100 * float(v[fits].mean()), 2) for k, v in per.items()},
         f"source>{MAX_SRC} tokens, truncated (n={int((~fits).sum())})": {k: round(100 * float(v[~fits].mean()), 2) for k, v in per.items()}}
diag = diagnostics(preds, refs)

with open(RESULTS / "eval_test_predictions.jsonl", "x") as f:
    for i in range(len(uids)):
        f.write(json.dumps({"uid": uids[i], "src_tokens_full": int(full_len[i]), "truncated": bool(not fits[i]),
                            "reference": refs[i], "prediction": preds[i], **{k: round(float(v[i]), 4) for k, v in per.items()}}) + "\n")
save_json("eval_test_metrics.json", {"n": len(preds), "rouge_x100": strat, "diagnostics": diag,
          "scoring": {"library": f"evaluate {evaluate.__version__} / rouge_score", "use_stemmer": True, "aggregation": "mean of per-example F1, no bootstrap",
                      "rougeLsum_sentence_split": "regex (?<=[.!?])\\s+ joined with newlines, applied to predictions and references"}})
save_json("eval_generation_config.json", {"hash": GEN_HASH, "config": gen_cfg.to_dict(), "gen_batch": GEN_BATCH, "precision": "fp32 weights, fp16 autocast"})
save_json("eval_metadata.json", {"checkpoint": best_name, "best_val_loss_recorded": EXPECTED_LOSS, "val_loss_recomputed": val_loss,
          "dataset_revision": json.load(open(LOGS_DIR / "dataset_version.json")).get("revision"), "seed": SEED,
          "max_source_len": MAX_SRC, "max_target_len": MAX_TGT, "versions": now, "gpu": torch.cuda.get_device_name(0),
          "generation_seconds": round(elapsed), "utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())})

rl = per["rougeL"]; picks = {}
for i in np.random.RandomState(SEED).choice(len(uids), 4, replace=False): picks.setdefault(int(i), "random")
for i in np.argsort(rl)[:4]: picks.setdefault(int(i), "lowest_rougeL")
for i in np.argsort(-full_len)[:2]: picks.setdefault(int(i), "longest_source")
save_json("eval_qualitative_examples.json", [{"group": g, "uid": uids[i], "src_tokens_full": int(full_len[i]), "rougeL": float(rl[i]),
          "transcript": srcs[i], "reference": refs[i], "prediction": preds[i]} for i, g in picks.items()])

print(json.dumps(strat, indent=2)); print("diagnostics:", diag); print(f"generation time: {elapsed:.0f}s")
for i, g in picks.items():
    print(f"\n[{g}] {uids[i]} | src_tokens={full_len[i]} | rougeL={rl[i]:.3f}")
    print(" SRC :", srcs[i][:200].replace("\n", " "), "…"); print(" REF :", refs[i][:300]); print(" PRED:", preds[i][:300])

{
  "all": {
    "rouge1": 71.3,
    "rouge2": 61.06,
    "rougeL": 68.47,
    "rougeLsum": 69.08
  },
  "source<=1024 tokens (n=416)": {
    "rouge1": 72.07,
    "rouge2": 61.92,
    "rougeL": 69.18,
    "rougeLsum": 69.69
  },
  "source>1024 tokens, truncated (n=446)": {
    "rouge1": 70.59,
    "rouge2": 60.27,
    "rougeL": 67.81,
    "rougeLsum": 68.51
  }
}
diagnostics: {'n': 862, 'empty_pred': 0, 'pred_tokens_mean/median': [63.4, 55.0], 'ref_tokens_mean/median': [82.5, 68.0], 'pred_near_max_len_pct': 0.1, 'pred_no_final_punct_pct': 41.5, 'pred_repeated_trigram_pct': 12.3, 'ref_repeated_trigram_pct': 44.2}
generation time: 627s

[random] LongBeachCC_02042014_14-0092 | src_tokens=696 | rougeL=0.853
 SRC : There are conditions on this one. This is recommendation to receive an f, receive the application of come and go market for an original ABC license at 33978 LAC Atlantic Avenue determine that the appl …
 REF : Recommendation to receive the application of Abdul Jamal Sheriff, dba 

In [ ]:
# SYNTHETIC OUT-OF-DOMAIN TRANSCRIPT -> FINE-TUNED BART -> GENERATED SUMMARY
# Inference only. Does not retrain, modify, or save model artifacts.

import os
import torch
from transformers import BartForConditionalGeneration, BartTokenizer

# 1. Load the BEST checkpoint (ARTIFACT_DIR is set in the evaluation section above)
CHECKPOINT = os.path.join(str(ARTIFACT_DIR), "checkpoints", "checkpoint-1938")

assert os.path.isfile(os.path.join(CHECKPOINT, "model.safetensors")), (
    f"Model weights not found at: {CHECKPOINT}"
)

tokenizer = BartTokenizer.from_pretrained(
    CHECKPOINT,
    local_files_only=True
)

model = BartForConditionalGeneration.from_pretrained(
    CHECKPOINT,
    local_files_only=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Loaded checkpoint:", CHECKPOINT)
print("Device:", device)

# 3. Scripted, chat-style example transcript (synthetic; NOT from MeetingBank and out-of-domain)
TRANSCRIPT = """
Alex Rivera: Alright team, thanks for hopping on. Today’s focus is simple: we need to finalize our budget projection and tier selection for integrating the Gemini API across our core application suite before next month's product rollout. Sarah, Marcus, how are we looking on the usage estimates?

Marcus Vance: Hey Alex. So, from the engineering side, we finished stress-testing the prototype. We're splitting our traffic into two distinct workloads. First, high-volume, low-latency background tasks like auto-categorization and quick inline text completions. For those, Gemini 2.5 Flash—or the newer Flash models—gives us more than enough reasoning power at a fraction of the cost per million tokens. Second, for complex document analysis and multi-step agentic workflows, we definitely need the flagship models like Gemini 2.5 Pro or Gemini 3.0 Pro.

Sarah Chen: Right, and that directly impacts how Google bills us. Under the Pay-As-You-Go pricing model via Google Cloud Vertex AI and Google AI Studio, input and output tokens are priced separately, and prompt caching plays a huge role. For our repeated system instructions and massive context windows—say, 100k+ tokens—cached input tokens cost roughly 75% less. If we structure our prompts carefully, our operational costs drop significantly.

Alex Rivera: That's huge. What about rate limits and quota tiers? Are we going to run into any throttling issues if we spike during peak user hours?

Marcus Vance: That depends on which plan tier we choose. On the Free Tier in AI Studio, you’re capped at a low RPM (Requests Per Minute) and TPM (Tokens Per Minute), plus your data can be used to improve Google products, so that's an automatic dealbreaker for production and user privacy. For commercial deployment, we need the Pay-As-You-Go Tier or an Enterprise Commitment Plan. Pay-As-You-Go lifts the rate limits substantially—giving us up to thousands of requests per minute depending on the model—and guarantees zero data logging or model training on our inputs.

Sarah Chen: Exactly. Furthermore, if we go through Vertex AI with Enterprise agreement, we can negotiate Provisioned Throughput. That allows us to reserve dedicated capacity (measured in Model Units), ensuring guaranteed latency and throughput regardless of global API traffic surges. The trade-off is a fixed monthly baseline cost rather than purely variable usage-based billing.

Alex Rivera: Understood. So to sum up our action items: Marcus, your team will refactor the system prompts to leverage Prompt Caching for the Pro model calls. Sarah, you’ll set up Google Cloud Billing budgets with automated alert thresholds—say at 50%, 75%, and 90% of our monthly cap—to prevent overages.

Marcus Vance: Sounds like a plan. We'll also route routine background requests to the Flash tier by default and reserve Pro for high-complexity fallback paths.

Sarah Chen: Perfect. I'll draft the cost projection breakdown comparing Pay-As-You-Go vs. Provisioned Throughput and share it in Slack by end of day.

Alex Rivera: Awesome. Thanks everyone, short and sweet. Catch you at standup tomorrow!"""

assert TRANSCRIPT.strip()

# 4. Tokenize and check whether the input will be truncated
full_tokens = tokenizer(
    TRANSCRIPT,
    add_special_tokens=True,
    truncation=False
)["input_ids"]

inputs = tokenizer(
    TRANSCRIPT,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

print(f"\nOriginal input tokens: {len(full_tokens)}")
print(f"Tokens given to model: {inputs['input_ids'].shape[1]}")

if len(full_tokens) > 1024:
    print("WARNING: Input truncated to 1024 tokens; only its beginning is summarized.")

# 5. Generate the summary using the evaluation generation settings
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        min_new_tokens=16,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True,
        do_sample=False,
        use_cache=True
    )

summary = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
).strip()

# 6. Display the actual result
print("\n" + "=" * 70)
print("INPUT TRANSCRIPT")
print("=" * 70)
print(TRANSCRIPT.strip())

print("\n" + "=" * 70)
print("GENERATED SUMMARY")
print("=" * 70)
print(summary)

print("\n" + "=" * 70)
print("SUMMARY COMPLETE")
print("=" * 70)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Loaded checkpoint: /content/drive/MyDrive/meetingbank_bart_backup_20261020_154746/checkpoints/checkpoint-1938
Device: cpu

Original input tokens: 689
Tokens given to model: 689


[transformers] Both `max_new_tokens` (=256) and `max_length`(=62) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=16) and `min_length`(=11) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



INPUT TRANSCRIPT
Alex Rivera: Alright team, thanks for hopping on. Today’s focus is simple: we need to finalize our budget projection and tier selection for integrating the Gemini API across our core application suite before next month's product rollout. Sarah, Marcus, how are we looking on the usage estimates?

Marcus Vance: Hey Alex. So, from the engineering side, we finished stress-testing the prototype. We're splitting our traffic into two distinct workloads. First, high-volume, low-latency background tasks like auto-categorization and quick inline text completions. For those, Gemini 2.5 Flash—or the newer Flash models—gives us more than enough reasoning power at a fraction of the cost per million tokens. Second, for complex document analysis and multi-step agentic workflows, we definitely need the flagship models like Gemini 2.5 Pro or Gemini 3.0 Pro.

Sarah Chen: Right, and that directly impacts how Google bills us. Under the Pay-As-You-Go pricing model via Google Cloud Vertex A

In [ ]:
# Example: first MeetingBank test item with 300-900 source tokens (not selected by score).
# NOTE: decoding here differs from the official evaluation (max_new_tokens=180, min_new_tokens=20, length_penalty=1.0).
# Requires cells above (tokenizer, model, device, LOGS_DIR).
from datasets import load_dataset
import torch

# Load the official MeetingBank test split at the pinned revision
ver = json.load(open(LOGS_DIR / "dataset_version.json"))
dataset = load_dataset(ver["dataset"], revision=ver["revision"], split="test")

# Find an example with a reasonably sized source transcript.
# Use the dataset's actual reference summary for comparison.
example = next(
    row for row in dataset
    if 300 <= len(tokenizer(
        row["transcript"],
        truncation=False
    )["input_ids"]) <= 900
)

transcript = example["transcript"]
reference = example["summary"]

inputs = tokenizer(
    transcript,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

# Remove inherited length settings that conflict with explicit settings.
generation_config = model.generation_config
generation_config.max_length = None
generation_config.min_length = 0

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=180,
        min_new_tokens=20,
        num_beams=4,
        length_penalty=1.0,
        no_repeat_ngram_size=3,
        early_stopping=True,
        do_sample=False,
    )

prediction = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
).strip()

print("=" * 70)
print("MEETING TRANSCRIPT (first 2500 characters)")
print("=" * 70)
print(transcript[:2500])

print("\n" + "=" * 70)
print("REFERENCE SUMMARY")
print("=" * 70)
print(reference)

print("\n" + "=" * 70)
print("MODEL-GENERATED SUMMARY")
print("=" * 70)
print(prediction)

print("\nInput tokens:", inputs["input_ids"].shape[1])
print("Generated tokens:", len(output_ids[0]))

README.md:   0%|          | 0.00/3.32k [00:00<?, ?B/s]

train.json: reconstructing file:   0%|          |  0.00B / 88.4MB            

train.json: downloading bytes:           |  0.00B            

validation.json: reconstructing file:   0%|          |  0.00B / 13.2MB            

validation.json: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/5169 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/861 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/862 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1338 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MEETING TRANSCRIPT (first 2500 characters)
Madam Court, could you please read docket 1239? Certainly. Docket 1239. The Committee on Government Operations, to which was referred on December 1st, 2021, docket number 1239 message an order authorizing the creation of a sheltered market program in conformity with the requirements of general laws. Chapter 30 B Section 18. This authorization applies to contracts for goods, professional services and support services. This authorization is for no more than six contracts, which must be awarded by June 30th, 2022. This sheltered market program shall be available for disadvantaged, minority and women only vendors, for whom there is a demonstrated substantial disparity in the city's 2020 disparities. Study submits a report recommending the order ought to pass. Thank you so much, Madam Clerk. The Chair recognizes Councilor Edwards, chair of the committee. Councilor Edwards. You have the floor. This is this is actually a matter, I believe, sponsored 